# Laboratorio 8 - Visión Por Computadora

Autores:

- Nelson García
- Joaquín Puente
- Diego Linares

Link al repositorio: https://github.com/Its-Japo/VisionXComputadora/tree/main/Lab_8

# Task 1

El siguiente conjunto de preguntas evalúa su capacidad de analizar, justificar y conectar los fundamentos
matemáticos de la detección de objetos con decisiones de ingeniería reales dentro del proyecto VisorShelf.
No basta con enunciar fórmulas: se espera que usted explique el significado de cada término y argumente
su relevancia práctica en el contexto de auditoría de anaqueles.

## Pregunta 1.1

El gerente de producto de VisorShelf le presenta la siguiente situación: el sistema detecta una lata de atún en el anaquel y devuelve la caja predicha b̂= (142, 89, 218, 165) en formato (x_min, y_min, x_max, y_max). El radiólogo de calidad del cliente anota manualmente la caja real b* = (138, 84, 222, 170). El cliente pregunta: "¿Qué tan buena es esa detección?"

#### 1) Cálculo manual del IoU

Se tiene:

Caja predicha:

b^ = (142,89,218,165)

Caja real (ground truth):

$b^∗$ = (138,84,222,170)

Formato: ($x_{min} , y_{min}, x_{max}, y_{max}$)

Paso 1: área de la caja predicha

Ancho: 218 − 142 = 76

Alto: 165 − 89 = 76

Área: ∣b^∣ = 76 * 76 = 5776

Paso 2: área de la caja real

Ancho: 222 − 138 =84

Alto: 170 − 84 = 86

Área: ∣$b^*$∣ = 84 * 86 = 7224

Paso 3: calcular la intersección

Para la intersección:

$x_{min}^I$ = max(142,138) = 142

$y_{min}^I$ = max(89,84) = 89

$x_{max}^I$ = min(218,222) = 218

$y_{max}^I$ = min(165,170) = 165

Entonces:

Ancho intersección: 218 − 142 = 76

Alto intersección: 165 − 89 = 76

Área de intersección: ∣I∣ = 76 * 76 = 5776


Paso 4: calcular la unión

Por inclusión-exclusión:

∣U∣= ∣b^∣ + ∣$b^∗$∣ − ∣I∣

Sustituyendo: ∣U∣ = 5776 + 7224 − 5776 = 7224

Paso 5: calcular IoU

IoU = ∣U∣/∣I∣ ​= 7224/5776 ​≈ 0.799

IoU ​≈ 0.80

Un IoU de 0.80 significa que la caja que predijo el sistema se superpone muy bien con la ubicación real del producto. En palabras simples: el sistema sí encontró la lata de atún y la marcó bastante cerca de donde realmente está. No es perfecta, pero para una tarea de auditoría de anaqueles es una detección buena y confiable. La definición formal de IoU y su cálculo por intersección/unión aparece en el material de detección del curso.

#### 2) 

En la fórmula:

IoU = ∣U∣ / ∣I∣
	​
∣I∣: área de intersección, es decir, la parte que comparten la caja predicha y la caja real.

∣U∣: área de unión, es decir, toda la región cubierta por cualquiera de las dos cajas.

¿Por qué usar la unión y no solo el área del ground truth?

Porque la unión penaliza tanto quedarse corto como pasarse:

- Si la predicción es muy pequeña, la intersección baja.
- Si la predicción es demasiado grande, la unión crece mucho.
- Si la caja está desplazada, la intersección cae y la unión sigue siendo grande.

Si el denominador fuera solo el área del ground truth, una caja muy grande que cubra el objeto y bastante fondo podría parecer artificialmente “buena”. Usar la unión evita premiar predicciones infladas o descuidadas. Ese diseño hace la métrica más justa y más robusta frente a detectores que “abrazan” demasiado contexto del anaquel.

Evita que el modelo gane puntuación simplemente dibujando cajas excesivamente grandes para asegurarse de tocar el producto. En VisorShelf eso sería grave, porque una caja demasiado amplia podría invadir productos vecinos y distorsionar decisiones de inventario, planograma o conteo frontal.

#### 3)

En un anaquel, la meta de negocio no es competir por localización “milimétrica”, sino detectar correctamente presencia, ausencia y ubicación razonable del producto. Un umbral de 0.5 suele aceptar detecciones útiles aunque la caja no sea perfecta; 0.75 es más estricto y exigiría una alineación muy fina. El material del curso justamente presenta 0.5 como umbral histórico y 0.75 como criterio más exigente.

Impacto operativo

Con θ=0.75: 
- aumentan los falsos negativos.
- una detección útil podría rechazarse por no ajustar con suficiente precisión.
- el sistema podría reportar quiebres de stock inexistentes o dejar productos “sin detectar”.

Con θ=0.5:
- se tolera una localización menos fina;
- baja el riesgo de perder productos presentes;
- es más adecuado cuando lo importante es auditoría de anaquel y no recorte exacto para robótica o inspección de alta precisión.


Recomendación práctica:

Producción: usar IoU = 0.5 como criterio de validez.
Evaluación interna adicional: también monitorear IoU más altos (por ejemplo 0.75) para medir calidad fina del posicionamiento.

En retail, un falso negativo puede ser costoso porque el sistema puede “ver vacío” donde sí hay producto, generando acciones innecesarias. Un falso positivo también molesta, pero normalmente es más manejable que perder visibilidad real sobre stock. Por eso 0.5 es mejor balance para este caso.

## Pregunta 1.2

Durante una prueba piloto en una tienda de conveniencia, el detector de VisorShelf analiza una imagen con 15 productos en el anaquel. El modelo genera 18 predicciones. Tras aplicar el umbral IoU = 0.5, el equipo clasifica: 12 TP, 6 FP y 3 FN:

Se reporta:

TP = 12
FP = 6
FN = 3


#### 4)

Precisión
P = TP / (TP + FP) = 12 / 12 + 6 = 12 / 18 = 0.667
P = 0.67 o 66.7%
	​

Recall
R = TP / TP + FN = 12 / 12 + 3 = 12 / 15 = 0.8
R=0.80 o 80%
	​

Explicación de cada denominador

En Precisión: TP + FP

representa todas las detecciones que el sistema afirmó como positivas. Es decir: todo lo que el sistema dijo “aquí hay un producto”.

TP: aciertos.
FP: falsas alarmas.

Entonces la Precisión responde: de todo lo que el sistema marcó, cuánto era correcto.

En Recall: TP + FN

representa todos los objetos reales que realmente estaban en la imagen.

TP: objetos reales detectados.
FN: objetos reales que el sistema no encontró.

Entonces el Recall responde: de todos los productos que sí existían, cuántos logró recuperar el sistema.

¿Por qué ambas métricas son necesarias?

Porque miden fallas distintas:

Precisión alta evita llenar el sistema de falsas alarmas.
Recall alto evita perder productos reales.

Un detector puede tener Recall alto detectando casi todo, pero generando muchas cajas erróneas. O puede tener Precisión alta siendo muy conservador, pero perdiendo muchos productos. En VisorShelf se necesitan ambas perspectivas para entender si el sistema sirve operativamente. 

#### 5)

La frase:

“Prefiero que el sistema no se pierda ningún quiebre de stock, aunque a veces nos avise falsas alarmas.”

se traduce a:

- priorizar Recall alto
- aceptar una Precisión más baja si es necesario

Ajustaría el umbral de confianza τ hacia abajo.

Porque al bajar τ: se aceptan más detecciones; suben los TP potenciales; baja el número de FN; mejora el Recall. El costo es que también pueden subir los FP, y por tanto bajar la Precisión.

Para operaciones, perder un quiebre real de stock puede implicar: venta perdida, reposición tardía, anaquel vacío sin atención.

En cambio, una falsa alarma genera revisión innecesaria, pero suele ser menos costosa que no detectar una ausencia real. Por eso la preferencia del director es claramente una preferencia por Recall.


#### 6)

El mAP (Mean Average Precision) es la métrica estándar para evaluar detectores de objetos. 

- 1. para cada clase se construye la curva Precisión–Recall variando el umbral de confianza τ;
- 2. se calcula el AP (Average Precision) como el área bajo esa curva;
- 3. luego se promedian los AP de todas las clases para obtener el mAP.

¿Por qué es más informativo?

Porque un único valor de Precisión o Recall depende de un solo punto operativo, es decir, de un solo umbral de confianza. En cambio, el mAP resume el comportamiento del detector a lo largo de muchos umbrales, mostrando mejor el trade-off real entre detectar más y equivocarse más.

Eso es importante en VisorShelf porque el sistema podría operar distinto según la política del cliente:

- una tienda puede preferir Recall alto.
- otra puede querer menos falsas alarmas.
- mAP permite comparar modelos de forma más completa.

mAP@0.5 (PASCAL VOC)

Usa un solo criterio de acierto:

IoU ≥ 0.5

Si la caja supera ese umbral, cuenta como detección correcta. Es una evaluación más permisiva respecto a la localización.

mAP@0.5:0.95 (COCO)

Promedia el AP sobre múltiples umbrales IoU:

0.50,0.55,0.60,…,0.95

Eso exige que el detector no solo encuentre el objeto, sino que además lo localice bien de forma consistente. El curso señala justamente esa diferencia entre VOC y COCO.

¿Cuál sería más exigente para VisorShelf?

El más exigente sería:

mAP@0.5:0.95
	​
porque castiga cajas imprecisas incluso cuando el objeto fue detectado “más o menos bien”. En anaqueles densos, donde productos vecinos son muy parecidos y están muy juntos, esa exigencia extra es relevante: una caja mal ajustada puede invadir el producto contiguo y afectar conteo, ubicación y análisis de planograma.

## Pregunta 1.3

En una imagen de anaquel con 40 productos, el modelo de VisorShelf genera 312 cajas candidatas antes
de cualquier postprocesamiento. El cliente observa el resultado intermedio y exclama: "¡El sistema está
viendo el mismo producto decenas de veces!"

#### 7)

El cliente dice:
“¡El sistema está viendo el mismo producto decenas de veces!”

Eso ocurre porque, antes del filtrado final, el detector suele proponer muchas cajas parecidas alrededor del mismo objeto. Esto es normal en detectores modernos, especialmente de una etapa, donde muchas celdas o anclas compiten por explicar la misma región.

El Non-Maximum Suppression (NMS) es el mecanismo que limpia ese resultado.

NMS significa, en esencia: “De varias cajas que parecen apuntar al mismo producto, me quedo con la mejor y elimino las repetidas.”

Paso a paso:
- 1. El sistema mira todas las cajas candidatas y sus puntajes de confianza.
- 2. Ordena las cajas desde la más confiable hasta la menos confiable.
- 3. Toma la mejor caja y la conserva.
- 4. Revisa las demás:
- - si una caja se superpone mucho con la ya elegida, asume que probablemente es el mismo producto;
entonces la elimina.
- - Repite el proceso con la siguiente caja sobreviviente.
- 5. Al final queda una sola caja por producto, o unas pocas bien separadas.

¿Por qué el detector genera múltiples cajas?

Porque durante la inferencia:
- varias anclas o posiciones distintas pueden responder al mismo producto.
- el modelo no “sabe” desde el inicio cuál de esas cajas es la definitiva.
- por eso propone varias hipótesis cercanas, y luego NMS hace la depuración final. El material del curso describe precisamente este problema de múltiples detecciones y el papel del NMS como refinamiento posterior.

#### 8)

Si el umbral NMS es bajo, el algoritmo se vuelve muy agresivo:

- ve dos cajas cercanas.
- detecta superposición moderada.
- y puede borrar una aunque en realidad correspondan a dos productos distintos uno al lado del otro.

Eso sería peligroso en anaqueles densos, porque podría reducir detecciones válidas y generar falsos negativos.

Riesgo de umbral bajo
- suprime cajas de productos vecinos reales.
- sube FN.
- el sistema puede “fusionar” dos productos contiguos en uno solo.

Riesgo de umbral alto
- deja pasar más duplicados.
- sube FP por detecciones repetidas del mismo objeto.

Para VisorShelf, en estantes apretados, conviene un NMS menos agresivo, o sea con umbral más alto, para no borrar productos verdaderos contiguos. Luego se puede controlar el exceso de duplicados combinándolo con un buen umbral de confianza y calibración del detector.

#### 9)

El orden correcto es:
1. Aplicar primero el umbral de confianza τ
2. Después aplicar NMS

El umbral de confianza elimina primero las cajas evidentemente débiles o improbables. Así, NMS trabaja sobre un conjunto mucho más pequeño y más limpio.

Eso tiene dos ventajas:

- computacional: menos cajas para comparar entre sí.
- estadística: menos ruido compitiendo con cajas buenas.

¿Qué pasaría si se invierte el orden?

Si se corre NMS antes de filtrar por confianza:

NMS tendría que procesar muchísimas cajas irrelevantes;
aumentaría el costo computacional;
algunas cajas de baja confianza podrían interferir innecesariamente en la supresión.

En el caso descrito:

- 312 cajas candidatas por imagen
- 30 imágenes por minuto
- Eso significa: 312 × 30 = 9360, cajas candidatas por minuto antes de postprocesamiento.

Como NMS compara superposición entre muchas cajas, hacerlo antes de filtrar por confianza encarece bastante el sistema. En un escenario on-premise con CPU media y presupuesto menor a 500 ms por imagen, ese orden sería mala decisión de ingeniería. El flujo típico de detección presentado en clase también ubica primero el filtrado por score y luego la supresión de redundancias.

# Task 2

Las siguientes preguntas evalúan su comprensión estratégica de la evolución de los detectores de dos
etapas y su capacidad de tomar decisiones de arquitectura justificadas dentro del contexto operativo de
VisorShelf. Se valorará la coherencia del argumento con las restricciones reales del sistema.

## Pregunta 2.1

El CTO de VisorShelf propone usar el detector original R-CNN (2014) para la primera versión del sistema. El
equipo de ingeniería calcula que con el dataset actual y una CPU de tienda, cada imagen tardaría
aproximadamente 45 segundos en procesarse. Con esto responda en su reporte: